In [1]:
import os
from ldaca.ldaca import LDaCA
# interactive UI widgets
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual


In [2]:
import getpass
BASE_URL="https://data.ldaca.edu.au/api"
API_TOKEN=getpass.getpass(prompt="API token")

API token ········


In [3]:
ldaca = LDaCA(
    url=BASE_URL,
    token=API_TOKEN
)

In [4]:
# Saves the ro-crate-metadata.json in the data_dir
ldaca.retrieve_collection(
    collection='arcp://name,hdl10.25911~m03c-yz22',
    collection_type='Collection',
    data_dir='data')

/Users/uqrsmi33/repo/github.com/Language-Research-Technology/notebook-sydney-speaks/.venv/lib/python3.14/site-packages/rocrate/rocrate.py:193: UserWarning: arcp://name,hdl10.25911~m03c-yz22/SydS looks like a data entity but it's not listed in the root dataset's hasPart
  warnings.warn(f"{entity['@id']} looks like a data entity but it's not listed in the root dataset's hasPart")
/Users/uqrsmi33/repo/github.com/Language-Research-Technology/notebook-sydney-speaks/.venv/lib/python3.14/site-packages/rocrate/rocrate.py:193: UserWarning: arcp://name,hdl10.25911~m03c-yz22/BCNT looks like a data entity but it's not listed in the root dataset's hasPart
  warnings.warn(f"{entity['@id']} looks like a data entity but it's not listed in the root dataset's hasPart")
/Users/uqrsmi33/repo/github.com/Language-Research-Technology/notebook-sydney-speaks/.venv/lib/python3.14/site-packages/rocrate/rocrate.py:193: UserWarning: arcp://name,hdl10.25911~m03c-yz22/SSDS looks like a data entity but it's not liste

In [5]:
speakers = [e.properties() for e in ldaca.crate.get_entities() if "speaker" in e.get("@id")]

ethnicities = set()
age_groups = set()
genders = set()
for speaker in speakers:
    ethnicities |= set(speaker.get('arcp://name,custom/terms#ethnicity', []))
    if type(speaker.get('arcp://name,custom/terms#ageGroup', [])[0]) == dict:
        as_strings = [a['@id'] for a in speaker.get('arcp://name,custom/terms#ageGroup', [])]
        age_groups |= set(as_strings)
    else:
        age_groups |= set(speaker.get('arcp://name,custom/terms#ageGroup', []))
    genders |= set(speaker.get('gender', []))

In [6]:
age_group_input = widgets.SelectMultiple(
    options=sorted(age_groups),
    value=[],
    description='Age group:',
    disabled=False,
)
ethnicity_input = widgets.SelectMultiple(
    options=sorted(ethnicities),
    value=[],
    description='Ethnicity:',
    disabled=False,
)
gender_input = widgets.SelectMultiple(
    options=sorted(genders),
    value=[],
    description='Gender:',
    disabled=False,
)

widgets.VBox(
    [
        widgets.Label(value='(Hold ctrl/cmd to select multiple)'), 
        age_group_input,
        ethnicity_input,
        gender_input
    ]
)

In [7]:
def get_audio(text_entity):
    parts = text_entity.get("hasPart")
    for part in parts:
        if  'audio/x-wav' in part.get('encodingFormat', []):
            return part
        elif part.id.endswith('.wav'):
            return part

def get_transcript(text_entity):
    parts = text_entity.get("hasPart")
    for part in parts:
        if part.id.endswith('.eaf'):
            return part

In [9]:
from lib import download_file
from IPython.display import display

download_audio_button = widgets.Button(description="Download audio")
download_transcripts_button = widgets.Button(description="Download transcripts")
output = widgets.Output()

texts = []
extensions = set()
def filter(_):
    global texts
    texts.clear()

    for e in ldaca.crate.get_entities():
        speaker = e.get("ldac:speaker", None)
        if speaker:
            speaker = speaker[0]
            ethnicity = speaker.get('arcp://name,custom/terms#ethnicity', [])
            age_group = speaker.get('arcp://name,custom/terms#ageGroup', [])
            gender = speaker.get('gender', [])
            if not set(age_group).isdisjoint(age_group_input.value) \
                and not set(ethnicity).isdisjoint(ethnicity_input.value) \
                and not set(gender).isdisjoint(gender_input.value):
                texts.append(e)
                parts = e.get("hasPart")
                for part in parts:
                    (_, ext) = os.path.splitext(part.id)
                    extensions.add(ext)
                    print(ext)
    with output:
        output.clear_output()
        display(widgets.VBox([make_file_button(ext) for ext in extensions]))
        print(f'Selected {len(texts)} objects')


def file_download_click_handler(ext):
    with output:
        for text_entity in texts:
            parts = text_entity.get("hasPart")
            for part in parts:
                if part.id.endswith(ext):
                    download_file(ldaca, part, API_TOKEN=API_TOKEN)

def make_file_button(ext):
    button = widgets.Button(description=f"Download {ext} files")
    button.on_click(lambda _b: file_download_click_handler(ext))
    return button

refresh_button = widgets.Button(
    description='Refresh',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='refresh' # (FontAwesome names without the `fa-` prefix)
)
refresh_button.on_click(filter)


display(widgets.VBox([
    refresh_button
]))
display(widgets.Box([]), output)

Box()

Output()

In [12]:
from IPython.display import FileLink
from pathlib import Path
from tqdm import tqdm
import zipfile
from shutil import make_archive

archive_name = f"sydney_speaks_subset"
make_archive(archive_name, 'zip', "./data")

display(FileLink(f'{archive_name}.zip'))

/Users/uqrsmi33/repo/github.com/Language-Research-Technology/notebook-sydney-speaks/sydney_speaks_subset.zip